# How to run the algorithm and experiments

This notebook provides a complete guide to using the Semi-Supervised Learning Imputation framework. 

* **Part 1** shows how to run the core algorithm (`UnlabeledLogReg`) step-by-step on a single dataset.
* **Part 2** provides a template for running the full, large-scale experimental suite on new data.
* **Part 3** contains a practical, reproducible example of the experiment using the DARWIN dataset.

# Necessary imports


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from pathlib import Path

import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from src.data_prep import ColumnSelector
from src.missing import MCAR, MAR1, MAR2, MNAR
from src.unlabeled_lr import UnlabeledLogReg
from src.experiment import run_experiment
from src.visualizations import plot_experiment_results, plot_sigma_results

warnings.filterwarnings("ignore")
RESULTS_PATH = Path("../results").resolve()

---
## Part 1: How to use the Unlabeled Logistic Regression Algorithm

If you want to use the algorithm independently of the experimental loop, follow these steps. The `UnlabeledLogReg` model expects a standard feature matrix `X` and a target vector `y` where missing labels are encoded as `-1`.

In [ ]:
# dataset = pd.read_csv("path_to_your_dataset.csv")

# Prepare feature matrix X (e.g., extract relevant columns, exclude target)
X = None # prepare according to comment above

# Prepare target vector y. 
# IMPORTANT: 'y' MUST be a 1D numpy array of type int, containing ONLY 0 and 1 values.
y = None # prepare according to comment above

# ---------------------- UNCOMMENT BELOW FOR QUICK CHECK ON DARWIN DATASET ----------------------

# darwin = pd.read_csv("../data/raw/DARWIN.csv")

# X = darwin.iloc[:, 1:-1].to_numpy()
# y = darwin.iloc[:, -1].to_numpy()

# y = np.where(y == "P", 1, 0)

# ------------------------------------------------------------------------------------------

# 2. TRAIN / VAL / TEST SPLIT
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train_true, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# 3. PREPROCESSING PIPELINE
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("selector", ColumnSelector(threshold=0.7)),
])

X_train_prep = pipeline.fit_transform(X_train, y_train_true)
X_val_prep = pipeline.transform(X_val)
X_test_prep = pipeline.transform(X_test)

# To use MAR/MNAR, we need X_train as a DataFrame
X_train_df = pd.DataFrame(X_train_prep)

y_train_df = pd.DataFrame({"Y_true_unobserved": y_train_true})
y_train_df["Y_observed"] = y_train_df["Y_true_unobserved"].copy().astype(int)
y_train_df["S"] = 0
y_train_df["Missing_Y"] = "no"

# ====================================================================
# [!] CHOOSE YOUR MISSING DATA MECHANISM:
# You can freely use MCAR, MAR1, MAR2, or MNAR. 
# Uncomment the one you want to test.
# ====================================================================

# Option A: MCAR (Missing Completely At Random)
y_missing_df = MCAR(y_train_df, p=0.5) 

# Option B: MAR1 (Missing At Random - Single Feature)
# random_col = np.random.choice(X_train_df.columns)
# y_missing_df = MAR1(X_train_df, y_train_df, feature_column=random_col, w=2.0, b=0.0)

# Option C: MAR2 (Missing At Random - All Features)
# y_missing_df = MAR2(X_train_df, y_train_df, W=np.ones(X_train_df.shape[1]), b=0.0)

# Option D: MNAR (Missing Not At Random)
# y_missing_df = MNAR(X_train_df, y_train_df, w_x=np.ones(X_train_df.shape[1]), w_y=-2.0, b=0.0)

# Extract the final observed column where missing labels are -1
y_train_obs = y_missing_df["Y_observed"].values 

# 5. INITIALIZE AND FIT THE MODEL
# You can choose: 'naive', 'pseudo_labels', 'iterative_pseudo_labels', 'self_training', 'label_propagation'
model = UnlabeledLogReg(
    y_imputation_method="self_training", 
    base_estimator=RandomForestClassifier(random_state=42),
    k_best=1
)

# Note: Passing y_train_true is optional, but it allows the model to track "Imputation_score" internally
print("Fitting model...")
model.fit(X_train_prep, y_train_obs, y_train_true)

# 6. VALIDATE AND PREDICT
print("Validating model...")
model.validate(X_val_prep, y_val, measure="balanced_accuracy")

print("Predicting on test set...")
y_pred = model.predict(X_test_prep)
y_prob = model.predict_proba(X_test_prep)

print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"Test ROC AUC: {roc_auc_score(y_test, y_prob):.3f}")

---
## Part 2: Template for Full Experiment on New Data

If you want to run the full benchmark (testing multiple mechanisms, seeds, and approaches), use the template below. Define the experimental constants, load your data, and ensure the target variable is purely binary (`0` and `1`).

In [ ]:
# ==========================================
# 1. PARAMETERS CONFIGURATION
# ==========================================
seeds = None # e.g., list(range(5))
approaches = None # e.g., ["naive", "self_training", "label_propagation"]
k_best = None # e.g., [1]

# Missing mechanism parameters
mar1_w, mar1_b = None, None
mar2_w, mar2_b = None, None
mnar_wx, mnar_wy, mnar_b = None, None, None

# ==========================================
# 2. DATA PREPARATION
# ==========================================
# dataset = pd.read_csv("path_to_your_dataset.csv")

# Prepare feature matrix X (e.g., extract relevant columns, exclude target)
X_exp = None

# Prepare target vector y. MUST be a 1D numpy array of type int (ONLY 0 and 1 values).
y_exp = None

# ==========================================
# 3. RUN EXPERIMENT
# ==========================================
# results_new_data = run_experiment(
#     X=X_exp, y=y_exp,
#     mar1_w=mar1_w, mar1_b=mar1_b,
#     mar2_w=mar2_w, mar2_b=mar2_b,
#     mnar_wx=mnar_wx, mnar_wy=mnar_wy, mnar_b=mnar_b,
#     seeds=seeds,
#     approaches=approaches,
#     k_best=k_best,
#     verbose=False,
#     save_path=RESULTS_PATH / 'name_of_experiment.csv'
# )

---
## Part 3: Practical Example (DARWIN Dataset)

Here is a complete, working example of the pipeline used on the DARWIN dataset. Notice how the categorical target variable ("P" and "H") is mapped to strictly `1` and `0`.

In [ ]:
# Constants for the experiment
seeds = list(range(10))
RESULTS_PATH = Path("../results").resolve()

approaches = ["naive", "self_training", "label_propagation"]
k_best = [1]

mar1_w = [1, 3]
mar1_b = [0, 2]
mar2_w = [1, 3]
mar2_b = [0, 2]
mnar_wx = [1, 3]
mnar_wy = [1, 3]
mnar_b = [0, 2]

darwin = pd.read_csv("../data/raw/DARWIN.csv")

X = darwin.iloc[:, 1:-1].to_numpy()
y = darwin.iloc[:, -1].to_numpy()

y = np.where(y == "P", 1, 0)

results_darwin = run_experiment(
    X=X,
    y=y,
    mar1_w=mar1_w,
    mar1_b=mar1_b,
    mar2_w=mar2_w,
    mar2_b=mar2_b,
    mnar_wx=mnar_wx,
    mnar_wy=mnar_wy,
    mnar_b=mnar_b,
    seeds=seeds,
    approaches=approaches,
    k_best=k_best,
    verbose=False,
    save_path=RESULTS_PATH / 'darwin.csv'
)